In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from popsim import param_utils
from popsim.modules.tearing import Island, Tearing, generate_disruption_phase_trajectory, generate_tearing_phase_trajectory

# Set up tearing module
dt = 1e-4 / 3  # s
time_base = param_utils.make_time_base(t0=0.0, t1=7.0, dt=dt)
tearing_config = Tearing.Config(
    magx_time=time_base
)

islands = [Island(3,2), Island(2,1)]

W = {island: 0.0 for island in islands}
F = {island: 0.0 for island in islands}
mode_phase={island: 0.0 for island in islands}

tearing_initial_state = Tearing.State(W=W, F=F, mode_phase=mode_phase)

rot_dur = 1.0
trigger_time = 5.5
disrupt_time = 6.5
dur_tq_to_spike = 1e-3
dur_cq = 10e-3
survival_time = 0.3
locking_dur = 0.2

tearing_params = Tearing.Params(
    rot_dur=1.0,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(
        disrupt_time, dur_tq_to_spike, time_base, dt
    ),
    tearing_phase=generate_tearing_phase_trajectory(
        trigger_time, rot_dur, locking_dur, time_base, dt
    ),
)

tearing_module = Tearing(config=tearing_config, islands=islands)

In [ ]:
import jax

from popsim.modules.magnetic_diagnostics import LowNArray, load_lown_config
from popsim.simulate import simulate

probe_connections, func_Bp_per_A = load_lown_config()

lown_array_config = LowNArray.Config(
    probe_connections=probe_connections,
    reconstructed_modes=[1,2,3]
)

lown_array_initial_state = LowNArray.State(tearing_state=tearing_initial_state)

lown_array_params = LowNArray.Params(
    tearing_params=tearing_params,
)

lown_array_module = LowNArray(config=lown_array_config, tearing_module=tearing_module, func_Bp_per_A=func_Bp_per_A)

jax.config.update("jax_platforms", "cpu")
lown_xarray = simulate(lown_array_module, time_base, lown_array_initial_state, lown_array_params)

In [ ]:
from popsim.visualize import visualize_time_series

#lown_xarray.to_netcdf("lown_simulation.nc")
visualize_time_series(lown_xarray, max_cols=2)